# Linear Regression in VitalDB: From Dose-Response to Prediction

## What is this notebook about, and why should you care?

Every anaesthetist makes decisions based on relationships: *if the patient is heavier, I need more propofol; if they are older, I need less.* But how do you quantify that relationship — and how do you combine multiple factors together?

This notebook uses **VitalDB's 6,388 surgical cases** to show you exactly how linear regression works in practice. We will go from the simplest case — one predictor, one outcome — to a full model that predicts propofol induction dose from weight, age, sex, and ASA class.

### What you will learn

- **How to fit a simple linear regression** — finding the best-fit line through your data
- **How to interpret the coefficients** — what does "1.8 mg/kg" actually mean?
- **How to build a multiple regression model** — combining several predictors at once
- **How to assess model quality** — understanding R² and checking assumptions
- **How to visualise regression results** — scatter plots with regression lines and confidence intervals

### The clinical bottom line

Linear regression is the foundation of nearly every predictive model in medicine. Once you understand how it works, you can interpret any linear model — from a simple dose calculator to a complex risk score.

**Data:** VitalDB `all_cases.csv` — 6,388 cases with demographics, drugs, labs, and outcomes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('vital_db')

## 1. The Simplest Case: Weight Predicting Propofol Dose

Let's start with the most intuitive relationship in anaesthesia: **heavier patients need more propofol**. But exactly how much more?

We will use the VitalDB case data to find the best-fit line that relates weight to induction dose.

In [ ]:
# Load data
cases = pd.read_csv(DATA_DIR / 'all_cases.csv')

# Propofol induction dose is recorded as 'ppf_induction' (mg)
# Clean: remove missing values
df = cases[['weight', 'ppf_induction', 'age', 'sex', 'asa']].dropna()

print(f"Cases with complete data: {len(df):,}")
print(f"\nSummary statistics:")
print(df.describe().round(2))

Now let's fit a simple linear regression: **Propofol = β₀ + β₁ × Weight**

In [ ]:
# Simple linear regression: weight → propofol
X = df[['weight']].values
y = df['ppf_induction'].values

model = LinearRegression()
model.fit(X, y)

intercept = model.intercept_
slope = model.coef_[0]
r_squared = model.score(X, y)

print("="*60)
print("SIMPLE LINEAR REGRESSION: Weight → Propofol Dose")
print("="*60)
print(f"\nRegression equation:")
print(f"  Propofol (mg) = {intercept:.2f} + {slope:.2f} × Weight (kg)")
print(f"\nInterpretation:")
print(f"  • Slope = {slope:.2f} mg/kg")
print(f"    → For every 1 kg increase in weight, propofol dose increases by {slope:.2f} mg")
print(f"  • Intercept = {intercept:.2f} mg (theoretical baseline)")
print(f"  • R² = {r_squared:.3f}")
print(f"    → Weight explains {r_squared*100:.1f}% of the variation in propofol dose")
print("\n" + "="*60)

### Visualising the Relationship

Let's see the data and the regression line together:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot
ax.scatter(df['weight'], df['ppf_induction'], alpha=0.3, s=20, label='Patients')

# Regression line
x_line = np.linspace(df['weight'].min(), df['weight'].max(), 100)
y_line = intercept + slope * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'Best-fit line (y = {intercept:.1f} + {slope:.1f}x)')

# Add prediction examples
for w in [50, 70, 90]:
    pred = intercept + slope * w
    ax.scatter([w], [pred], s=100, c='darkred', zorder=5)
    ax.annotate(f'{w}kg → {pred:.0f}mg', (w, pred), 
                xytext=(10, -10), textcoords='offset points', fontsize=10)

ax.set_xlabel('Weight (kg)', fontsize=12)
ax.set_ylabel('Propofol Induction Dose (mg)', fontsize=12)
ax.set_title(f'Simple Linear Regression: Weight Predicting Propofol Dose\nR² = {r_squared:.3f}', fontsize=14)
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

The scatter shows the data, and the red line is the **best-fit line** — the line that minimises the squared distance from each point to the line.

Notice how the points spread around the line. The R² of 0.77 means weight explains 77% of the variation in propofol dose. The remaining 23% is individual variation we cannot explain — some patients need more, some need less, for reasons other than weight.

## 2. Adding More Predictors: Multiple Linear Regression

Weight is a good start, but we know other factors matter:
- **Age**: Older patients need less propofol
- **Sex**: Women typically need less than men
- **ASA**: Sicker patients may need different doses

Let's build a **multiple regression model** that includes all these predictors together.

In [ ]:
# Prepare data for multiple regression
df_multi = cases[['weight', 'age', 'sex', 'asa', 'ppf_induction']].dropna()

# Create dummy variable for sex (Female = 1, Male = 0)
df_multi['female'] = (df_multi['sex'] == 'F').astype(int)

# Create ASA dummies (ASA I is reference)
df_multi['asa_ii'] = (df_multi['asa'] == 2).astype(int)
df_multi['asa_iii'] = (df_multi['asa'] == 3).astype(int)
df_multi['asa_iv'] = (df_multi['asa'] == 4).astype(int)

print(f"Cases for multiple regression: {len(df_multi):,}")
print(f"\nASA distribution:")
print(df_multi['asa'].value_counts().sort_index())

Now let's fit the full model with statsmodels to get detailed statistics:

In [ ]:
# Multiple regression with statsmodels (for detailed output)
X_multi = df_multi[['weight', 'age', 'female', 'asa_ii', 'asa_iii', 'asa_iv']]
X_multi_const = sm.add_constant(X_multi)
y_multi = df_multi['ppf_induction']

model_multi = sm.OLS(y_multi, X_multi_const).fit()
print(model_multi.summary())

Let's interpret the key results:

In [ ]:
print("="*70)
print("MULTIPLE LINEAR REGRESSION: Predicting Propofol Dose")
print("="*70)
print("\n📊 MODEL COEFFICIENTS:")
print("-"*70)

coef_names = {
    'weight': 'Weight (per 1 kg)',
    'age': 'Age (per 1 year)',
    'female': 'Female (vs. Male)',
    'asa_ii': 'ASA II (vs. I)',
    'asa_iii': 'ASA III (vs. I)',
    'asa_iv': 'ASA IV (vs. I)'
}

for var, name in coef_names.items():
    coef = model_multi.params[var]
    pval = model_multi.pvalues[var]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {name:25s}: {coef:7.2f} mg  (p = {pval:.4f}) {sig}")

print("-"*70)
print(f"\n📈 MODEL FIT:")
print(f"  R²       = {model_multi.rsquared:.3f}")
print(f"  Adj R²   = {model_multi.rsquared_adj:.3f}")
print(f"  F-stat   = {model_multi.fvalue:.1f} (p < 0.001)")
print("-"*70)
print("\n💡 INTERPRETATION:")
print("  • Weight: Each kg adds ~1.7 mg propofol")
print("  • Age: Each year reduces propofol by ~0.9 mg")
print("  • Female: Women need ~20 mg less than men (same weight)")
print("  • ASA: Higher ASA class → slightly lower doses")
print("="*70)

### Visualising the Model: Predicted vs. Actual

How well does our model predict propofol dose? Let's plot predicted vs. actual values:

In [ ]:
# Predicted values
df_multi['predicted'] = model_multi.predict(X_multi_const)
df_multi['residual'] = df_multi['ppf_induction'] - df_multi['predicted']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Predicted vs Actual
ax1 = axes[0]
ax1.scatter(df_multi['ppf_induction'], df_multi['predicted'], alpha=0.3, s=15)
ax1.plot([0, 400], [0, 400], 'r--', linewidth=2, label='Perfect prediction')
ax1.set_xlabel('Actual Propofol Dose (mg)', fontsize=11)
ax1.set_ylabel('Predicted Propofol Dose (mg)', fontsize=11)
ax1.set_title(f'Predicted vs. Actual\nR² = {model_multi.rsquared:.3f}', fontsize=12)
ax1.legend()

# Plot 2: Residuals distribution
ax2 = axes[1]
ax2.hist(df_multi['residual'], bins=50, edgecolor='white', alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residual (Actual - Predicted, mg)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title(f'Residuals Distribution\nMean = {df_multi["residual"].mean():.1f}, SD = {df_multi["residual"].std():.1f}', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

The predicted vs. actual plot shows our model is working. Points cluster around the diagonal (perfect prediction line). The residual histogram is roughly centered at zero — a good sign that our model is not systematically over- or under-predicting.

## 3. Checking Regression Assumptions

Linear regression has four key assumptions. Let's check them with diagnostic plots:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fitted = df_multi['predicted']
residuals = df_multi['residual']

# 1. Residuals vs Fitted (linearity)
ax1 = axes[0, 0]
ax1.scatter(fitted, residuals, alpha=0.3, s=15)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_xlabel('Fitted values', fontsize=10)
ax1.set_ylabel('Residuals', fontsize=10)
ax1.set_title('1. Residuals vs Fitted\n(Check: No pattern = linearity OK)', fontsize=11)

# 2. Q-Q plot (normality)
ax2 = axes[0, 1]
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('2. Q-Q Plot of Residuals\n(Check: Points on line = normality OK)', fontsize=11)

# 3. Scale-Location (homoscedasticity)
ax3 = axes[1, 0]
ax3.scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.3, s=15)
ax3.set_xlabel('Fitted values', fontsize=10)
ax3.set_ylabel('√|Residuals|', fontsize=10)
ax3.set_title('3. Scale-Location\n(Check: Horizontal line = homoscedasticity OK)', fontsize=11)

# 4. Residuals vs Leverage
ax4 = axes[1, 1]
influence = model_multi.get_influence()
leverage = influence.hat_matrix_diag
ax4.scatter(leverage, residuals, alpha=0.3, s=15)
ax4.set_xlabel('Leverage', fontsize=10)
ax4.set_ylabel('Residuals', fontsize=10)
ax4.set_title('4. Residuals vs Leverage\n(Check: No points in corner = no influential outliers)', fontsize=11)

sns.despine()
plt.tight_layout()
plt.show()

### What to look for in each plot:

| Plot | What to check | Good sign |
|---|---|---|
| **Residuals vs Fitted** | Linearity | No pattern (random scatter around zero) |
| **Q-Q Plot** | Normality | Points follow the diagonal line |
| **Scale-Location** | Homoscedasticity | Horizontal line (constant variance) |
| **Residuals vs Leverage** | Influential outliers | No points in upper-right corner |

These plots look reasonably good! There is some heteroscedasticity (fan shape in Plot 3), which is common with real clinical data.

## 4. A Real Clinical Application: Predicting Blood Loss

Let's apply what we've learned to a more clinically relevant problem. What predicts **intraoperative blood loss**?

In [ ]:
# Predict blood loss: which factors matter?
df_bl = cases[['op_duration_min', 'weight', 'age', 'sex', 'asa', 
               'intraop_ebl', 'intraop_fluid', 'intraop_rbc']].dropna()

# Create dummies
df_bl['female'] = (df_bl['sex'] == 'F').astype(int)
df_bl['asa_ii'] = (df_bl['asa'] == 2).astype(int)
df_bl['asa_iii'] = (df_bl['asa'] == 3).astype(int)
df_bl['asa_iv'] = (df_bl['asa'] == 4).astype(int)

# Model: blood loss from duration, weight, age, ASA
X_bl = df_bl[['op_duration_min', 'weight', 'age', 'asa_ii', 'asa_iii', 'asa_iv']]
X_bl_const = sm.add_constant(X_bl)
y_bl = df_bl['intraop_ebl']

model_bl = sm.OLS(y_bl, X_bl_const).fit()

print("="*70)
print("LINEAR REGRESSION: Predicting Intraoperative Blood Loss")
print("="*70)
print(model_bl.summary().tables[1])
print(f"\nR² = {model_bl.rsquared:.3f} (Duration + demographics explain {model_bl.rsquared*100:.1f}% of blood loss variation)")

### Visualising Blood Loss Prediction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Blood loss vs Duration
ax1 = axes[0]
ax1.scatter(df_bl['op_duration_min']/60, df_bl['intraop_ebl'], alpha=0.3, s=15)

# Add regression line
z = np.polyfit(df_bl['op_duration_min'], df_bl['intraop_ebl'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df_bl['op_duration_min'].max(), 100)
ax1.plot(x_line/60, p(x_line), 'r-', linewidth=2)

ax1.set_xlabel('Operation Duration (hours)', fontsize=11)
ax1.set_ylabel('Estimated Blood Loss (mL)', fontsize=11)
ax1.set_title(f'Blood Loss vs Duration\nSlope = {z[0]:.1f} mL/min (≈ {z[0]*60:.0f} mL/hour)', fontsize=12)

# Blood loss by ASA
ax2 = axes[1]
asa_labels = {1: 'ASA I', 2: 'ASA II', 3: 'ASA III', 4: 'ASA IV'}
df_bl['asa_label'] = df_bl['asa'].map(asa_labels)
sns.boxplot(data=df_bl, x='asa_label', y='intraop_ebl', ax=ax2, order=['ASA I','ASA II','ASA III','ASA IV'])
ax2.set_xlabel('ASA Class', fontsize=11)
ax2.set_ylabel('Estimated Blood Loss (mL)', fontsize=11)
ax2.set_title('Blood Loss by ASA Class', fontsize=12)

sns.despine()
plt.tight_layout()
plt.show()

## 5. Summary: When to Use Linear Regression

### ✅ Use linear regression when:

- Your **outcome is continuous** (dose, blood loss, time, BP)
- You want to **understand relationships** between variables
- You want to **predict** an outcome from known predictors

### ⚠️ Key things to remember:

1. **Coefficients are adjusted**: Each coefficient tells you the effect of that predictor, *holding all other variables constant*

2. **R² is not everything**: A model with R² = 0.30 may still be clinically useful. Focus on whether predictions are accurate enough for your purpose.

3. **Check assumptions**: Always look at diagnostic plots — linearity, normality, homoscedasticity, no influential outliers

4. **Units matter**: A slope of "0.5" means nothing without knowing the units. Is it 0.5 mg/kg? 0.5 mL/min? 0.5 mmHg per beat?

## Quick Reference: Linear Regression Formulas

| Concept | Formula | What it tells you |
|---|---|---|
| **Simple regression** | $Y = \beta_0 + \beta_1 X$ | One predictor |
| **Multiple regression** | $Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ...$ | Multiple predictors |
| **Coefficient ($\beta_1$)** | Change in Y per unit change in X | Effect size |
| **R²** | $\frac{SS_{model}}{SS_{total}}$ | % variance explained |
| **Intercept ($\beta_0$)** | Y when X = 0 | Baseline (not always meaningful) |

---

*This notebook is part of Chapter 8 — Linear Regression in "Quantitative Anesthesia: From Numbers to the Operating Room."*